# Tests: `fastermodels.gate` (source `nbs/03_gate.ipynb`)

In [ ]:
from fastcore.test import *
import sys, tempfile
from pathlib import Path

import torch
from safetensors.torch import load_file, save_file
from torchvision.models import resnet18

import fastermodels
from fastermodels.card import render_card
from fastermodels.gate import BATCH_TOL, GateRow, gate_passed, run_gate
from fastermodels.model import FasterModel, state_hash

In [ ]:
_ROOT = str(Path(fastermodels.__file__).parents[1])  # the only PYTHONPATH the reload is given


def write_artifact(d):
    "Write a complete, valid artifact directory and the manifest that describes it"
    torch.manual_seed(0)
    model = resnet18(num_classes=10, weights=None).eval()
    fm = FasterModel.wrap(model, 'torchvision.models.resnet18', {'num_classes': 10, 'weights': None},
                          recipe={'prune': 'none, this is a pipeline test'})
    fm.save_pretrained(d)
    row = dict(artifact='FP32', file='model.safetensors', params=11_181_642, bytes=44_726_568,
               macs=1_824_000_000, peak_activation_bytes=3_211_264,
               k=89, n=100, delta=-1.0, lo=-2.5, hi=0.5, p_mcnemar=0.32)
    Path(d, 'README.md').write_text(render_card(dict(
        name='test-resnet18', base_model='torchvision/resnet18', license='bsd-3-clause',
        datasets=['frgfm/imagenette'], tags=['fasterai'],
        scope_line='Untrained weights, n=100; pipeline evidence, not a published claim.',
        input_shape='3x64x64', recipe={'prune': 'none'},
        reference={'name': 'resnet18', 'k': 90, 'n': 100, 'bytes': 44_726_568, 'params': 11_181_642,
                   'macs': 1_824_000_000, 'peak_activation_bytes': 3_211_264},
        rows=[row], latency=None,
        provenance={'fastermodels': fastermodels.__version__, 'torch': torch.__version__,
                    'measured_on': '2026-09-11'})))
    return dict(license={'id': 'bsd-3-clause', 'validated_by': 'the publisher'},
                hashes={'safetensors': state_hash(fm)},
                parity=[{'arms': ['in-memory', 'safetensors'], 'agreement': 1.0, 'kind': 'same-precision'},
                        {'arms': ['batch 1', 'batch 32'], 'max_abs_diff': 1e-7, 'kind': 'batch-invariance'},
                        {'arms': ['in-memory', 'onnx'], 'agreement': 0.96, 'kind': 'cross-precision'}],
                delta={'delta': -1.0, 'lo': -2.5, 'hi': 0.5, 'floor': -3.0},
                files={}, head={'num_classes': 10, 'expected': 10}, widths={'conv1': 64, 'fc': 10},
                rows=[row], latency_rows='non mesurée', clean_reload=None, proof=None)


def verdicts(d, manifest):
    "Condition number -> verdict, for a gate run on `d`"
    return {r.condition: r.passed for r in run_gate(d, manifest, python=sys.executable, pythonpath=_ROOT)}

In [ ]:
_tmp = tempfile.TemporaryDirectory()
_d = _tmp.name
_manifest = write_artifact(_d)

_rows = run_gate(_d, _manifest, python=sys.executable, pythonpath=_ROOT)
test_eq(len(_rows), 10)
test_eq([r.condition for r in _rows], list(range(10)))
test_eq([r.name for r in _rows if r.condition == 3], ['accuracy delta measured'])
test_eq(type(_rows[0]), GateRow)
assert all(r.evidence and '\n' not in r.evidence for r in _rows)

# everything passes but the clean machine, which was not run
_v = {r.condition: r.passed for r in _rows}
test_eq([c for c, ok in _v.items() if not ok], [8])
test_eq([r.evidence for r in _rows if r.condition == 8], ['not run'])
test_eq(gate_passed(_rows), False)

# the reload really ran in another interpreter and got the published hash
assert state_hash(FasterModel.from_pretrained(_d)) in [r.evidence for r in _rows if r.condition == 1][0]

# with the clean-machine reload recorded, the artifact is publishable
_manifest['clean_reload'] = {'passed': True, 'machine': 'fresh venv'}
test_eq(gate_passed(run_gate(_d, _manifest, python=sys.executable, pythonpath=_ROOT)), True)

In [ ]:
# an INT8-only artifact publishes no safetensors: condition 1 reloads the TorchScript, runs it, and
# compares the file digest the manifest published
import hashlib

with tempfile.TemporaryDirectory() as _ts:
    _tiny = torch.nn.Sequential(torch.nn.Conv2d(3, 2, 3), torch.nn.Flatten(), torch.nn.Linear(2 * 6 * 6, 2)).eval()
    _pt = Path(_ts, 'model.torchscript.pt')
    torch.jit.save(torch.jit.trace(_tiny, torch.randn(1, 3, 8, 8)), str(_pt))
    _digest = hashlib.sha256(_pt.read_bytes()).hexdigest()
    _int8 = {**_manifest, 'hashes': {'torchscript': _digest}, 'input_shape': '3x8x8'}

    _row1 = [r for r in run_gate(_ts, _int8, python=sys.executable, pythonpath=_ROOT) if r.condition == 1][0]
    test_eq(_row1.passed, True)
    assert _row1.evidence.startswith(f'torchscript reloaded {_digest}'), _row1.evidence

    # one byte of the published file changed is a different artifact
    _bytes = bytearray(_pt.read_bytes()); _bytes[-1] ^= 1
    _pt.write_bytes(bytes(_bytes))
    test_eq([r for r in run_gate(_ts, _int8, python=sys.executable, pythonpath=_ROOT)
             if r.condition == 1][0].passed, False)
    _pt.unlink()

    # nothing to reload at all is a failure naming both files
    _none = [r for r in run_gate(_ts, _int8, python=sys.executable, pythonpath=_ROOT) if r.condition == 1][0]
    test_eq(_none.passed, False)
    assert 'model.safetensors' in _none.evidence and 'model.torchscript.pt' in _none.evidence, _none.evidence

In [ ]:
# a claim that is not backed by the artifact never passes
test_eq(verdicts(_d, {**_manifest, 'license': {'id': 'bsd-3-clause', 'validated_by': ''}})[0], False)
test_eq(verdicts(_d, {**_manifest, 'hashes': {'safetensors': 'deadbeef'}})[1], False)
test_eq(verdicts(_d, {**_manifest, 'parity': [{'arms': ['a', 'b'], 'agreement': 0.99, 'kind': 'same-precision'}]})[2], False)
test_eq(verdicts(_d, {**_manifest, 'parity': [{'arms': ['a', 'b'], 'agreement': 0.5, 'kind': 'cross-precision'}]})[2], False)

# a model whose output depends on the batch it is served in is not the model that was measured
_same = [p for p in _manifest['parity'] if p['kind'] == 'same-precision']
test_eq(verdicts(_d, {**_manifest, 'parity': _same + [{'arms': ['batch 1', 'batch 32'], 'max_abs_diff': 1e-2,
                                                       'kind': 'batch-invariance'}]})[2], False)
# 1e-4 is float32 kernel noise across batch sizes, not a model that changes with the batch
test_eq(verdicts(_d, {**_manifest, 'parity': _same + [{'arms': ['batch 1', 'batch 32'], 'max_abs_diff': 1e-4,
                                                       'kind': 'batch-invariance'}]})[2], True)
test_eq(BATCH_TOL, 1e-3)
test_eq(verdicts(_d, {**_manifest, 'parity': _same + [{'arms': ['batch 1', 'batch 32'],
                                                       'kind': 'batch-invariance'}]})[2], False)

# every row carries size, memory and MACs, and the evidence names the criterion a row does not
_no_macs = {k: v for k, v in _manifest['rows'][0].items() if k != 'macs'}
_row6 = [r for r in run_gate(_d, {**_manifest, 'rows': [_no_macs]}, python=sys.executable, pythonpath=_ROOT)
         if r.condition == 6][0]
test_eq(_row6.passed, False)
test_eq(_row6.evidence, 'row 0 has no macs')
test_eq(verdicts(_d, {**_manifest, 'rows': [{**_manifest['rows'][0], 'peak_activation_bytes': -1}]})[6], False)
test_eq(verdicts(_d, {**_manifest, 'rows': []})[6], False)

# the producer names the arm it judged — the published row with the lowest lower bound — and the gate
# repeats it first, without that changing the verdict
_named = {**_manifest['delta'], 'arms': 'INT8 ONNX (onnxruntime) vs reference'}
_row3 = [r for r in run_gate(_d, {**_manifest, 'delta': _named}, python=sys.executable, pythonpath=_ROOT)
         if r.condition == 3][0]
assert _row3.evidence.startswith('arms=INT8 ONNX (onnxruntime) vs reference delta='), _row3.evidence
test_eq(_row3.passed, verdicts(_d, _manifest)[3])


def _evidence3(delta):
    "Condition 3 as the gate reports it for this delta"
    return [r for r in run_gate(_d, {**_manifest, 'delta': delta}, python=sys.executable, pythonpath=_ROOT)
            if r.condition == 3][0]


# the target is reported, not judged: a variant that misses it is measured, so it publishes and says so
_missed = _evidence3({'delta': -4.0, 'lo': -5.0, 'hi': -3.0, 'target': -3.0})
test_eq(_missed.passed, True)
assert _missed.evidence.endswith('target=-3.0 not demonstrated'), _missed.evidence
assert _evidence3({'delta': -1.0, 'lo': -2.5, 'hi': 0.5, 'target': -3.0}).evidence.endswith('target=-3.0 met')

# `floor` is the name this key used to have, and a delta with no target is measured all the same
test_eq(_evidence3({'delta': -4.0, 'lo': -5.0, 'hi': -3.0, 'floor': -3.0}).evidence, _missed.evidence)
_no_target = _evidence3({'delta': -1.0, 'lo': -2.5, 'hi': 0.5})
test_eq(_no_target.passed, True)
assert 'target' not in _no_target.evidence, _no_target.evidence

# what was never measured is never a pass: no interval, or an interval that is not a number
test_eq(_evidence3({'delta': -1.0, 'hi': 0.5, 'target': -3.0}).passed, False)
test_eq(_evidence3({'delta': -1.0, 'lo': float('nan'), 'hi': 0.5, 'target': -3.0}).passed, False)
test_eq(_evidence3({'delta': -1.0, 'lo': float('-inf'), 'hi': 0.5}).passed, False)
assert _evidence3({'delta': -1.0, 'hi': 0.5}).evidence.endswith('no interval, so no verdict')
test_eq(verdicts(_d, {**_manifest, 'clean_reload': {'passed': False}})[8], False)

# a head that is not the head the manifest expects, and widths nobody recorded
test_eq(verdicts(_d, {**_manifest, 'head': {'num_classes': 1000, 'expected': 10}})[5], False)
test_eq(verdicts(_d, {**_manifest, 'widths': {}})[5], False)

# latency rows without a proof measured on the target
test_eq(verdicts(_d, {**_manifest, 'latency_rows': []})[6], False)
test_eq(verdicts(_d, {**_manifest, 'latency_rows': [{'device': 'Orin', 'median_ms': 0.5}]})[6], True)
test_eq(verdicts(_d, {**_manifest, 'latency_rows': [{'device': 'Orin', 'median_ms': 0.5}]})[9], False)
test_eq(verdicts(_d, {**_manifest, 'latency_rows': [{'device': 'Orin', 'median_ms': 0.5}], 'proof': {'engine': 'tensorrt'}})[9], True)

In [ ]:
# Condition 4 reads the exported file back: an ONNX in the directory is checked, not trusted
_net = torch.nn.Sequential(torch.nn.Conv2d(3, 4, 3), torch.nn.ReLU()).eval()
torch.onnx.export(_net, torch.randn(1, 3, 8, 8), str(Path(_d, 'model.onnx')), dynamo=False, opset_version=17,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})

test_eq(verdicts(_d, {**_manifest, 'files': {'onnx': {'opset': 17, 'dynamic_batch': True, 'n_q': 0, 'n_dq': 0}}})[4], True)

# what the manifest claims does not override what the file says
test_eq(verdicts(_d, {**_manifest, 'files': {'onnx': {'opset': 13, 'dynamic_batch': True, 'n_q': 0, 'n_dq': 0}}})[4], True)
test_eq(verdicts(_d, {**_manifest, 'files': {'onnx': {'opset': 17, 'dynamic_batch': True, 'n_q': 42, 'n_dq': 42}}})[4], False)
test_eq(verdicts(_d, {**_manifest, 'files': {}})[4], False)

# a static batch is refused
torch.onnx.export(_net, torch.randn(1, 3, 8, 8), str(Path(_d, 'model.onnx')), dynamo=False, opset_version=17,
                  input_names=['input'], output_names=['output'])
test_eq(verdicts(_d, {**_manifest, 'files': {'onnx': {'opset': 17, 'dynamic_batch': True, 'n_q': 0, 'n_dq': 0}}})[4], False)

# an ONNX nobody can read back is not a pass: neither a file the manifest claims and that is not there,
# nor a file that cannot be opened because onnx is not installed
_claimed = {**_manifest, 'files': {'onnx': {'opset': 17, 'dynamic_batch': True, 'n_q': 0, 'n_dq': 0}}}
Path(_d, 'model.onnx').unlink()
_row = [r for r in run_gate(_d, _claimed, python=sys.executable, pythonpath=_ROOT) if r.condition == 4][0]
test_eq(_row.passed, False)
test_eq(_row.evidence, 'model.onnx claimed in the manifest but missing')

torch.onnx.export(_net, torch.randn(1, 3, 8, 8), str(Path(_d, 'model.onnx')), dynamo=False, opset_version=17,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})
_saved = sys.modules.get('onnx')
sys.modules['onnx'] = None   # makes `import onnx` raise ImportError
try:
    _row = [r for r in run_gate(_d, _claimed, python=sys.executable, pythonpath=_ROOT) if r.condition == 4][0]
finally:
    if _saved is None: del sys.modules['onnx']
    else: sys.modules['onnx'] = _saved
test_eq(_row.passed, False)
test_eq(_row.evidence, 'install onnx to verify model.onnx')
Path(_d, 'model.onnx').unlink()

In [ ]:
# a card that claims more than it measured is refused
_card = Path(_d, 'README.md')
_clean = _card.read_text()
_card.write_text(_clean + '\nThis compression is lossless.\n')
test_eq(verdicts(_d, _manifest)[7], False)
_card.write_text(_clean)
test_eq(verdicts(_d, _manifest)[7], True)
_card.unlink()
test_eq(verdicts(_d, _manifest)[7], False)
_card.write_text(_clean)

# weights that are not the weights the manifest hashed
_sd = load_file(Path(_d, 'model.safetensors'))
_sd['net.fc.bias'] = _sd['net.fc.bias'] + 1
save_file(_sd, str(Path(_d, 'model.safetensors')))
test_eq(verdicts(_d, _manifest)[1], False)

# an artifact that does not reload at all is a failure, not an exception
test_eq(verdicts(_d, {**_manifest, 'hashes': {'safetensors': 'x'}})[1], False)
_tmp.cleanup()